# Exercício 11 — Pydantic (validação e dados estruturados)

**Objetivo:** dominar **Pydantic v2** para dados tipados em pipelines de IA — desde APIs até saídas estruturadas de LLMs.

**Deste notebook:** modelos base, `Field`, `field_validator`, `model_validator`, modelos aninhados, serialização JSON, **JSON Schema** e (opcional) **`with_structured_output`** com Gemini.

**Documentação:** pasta [`docs/`](docs/).

## 0. Ambiente

A maior parte das células **não precisa de rede**. A última secção usa Gemini — define `GOOGLE_API_KEY` no `.env` na **raiz do repositório**.

In [ ]:
from pathlib import Path
import os

from dotenv import load_dotenv

EX_ROOT = Path.cwd().resolve()
REPO_ROOT = EX_ROOT.parent.parent

load_dotenv(REPO_ROOT / ".env", override=False)
load_dotenv(EX_ROOT / ".env", override=True)

print("OK — exercício:", EX_ROOT)

## 1. `BaseModel` e `Field`

Um modelo declara **campos com tipos**. O Pydantic **coage** valores (ex.: `"42"` → `int`) quando faz sentido e rejeita o resto com `ValidationError`.

- `Field(..., description=...)` documenta o campo (aparece no JSON Schema — útil para LLMs).
- Restrições numéricas: `ge`, `le`, `gt`, etc.

In [ ]:
from pydantic import BaseModel, Field, ValidationError


class FichaPedagogica(BaseModel):
    """Exemplo fictício — curso interno (sem dados reais de saúde)."""

    codigo_aluno: str = Field(min_length=3, description="Identificador interno")
    horas_estudo_semana: int = Field(ge=0, le=60, description="Horas declaradas pelo formador")
    nota_quiz: float = Field(ge=0.0, le=20.0)


a = FichaPedagogica(codigo_aluno="AL-901", horas_estudo_semana=8, nota_quiz=15.5)
print(a)

try:
    FichaPedagogica(codigo_aluno="AB", horas_estudo_semana=8, nota_quiz=15.5)
except ValidationError as e:
    print("Erro esperado (codigo curto):", e.error_count(), "falhas")

## 2. `field_validator` — normalizar um campo

`mode='before'` corre **antes** da coerção de tipo (útil para limpar strings). `mode='after'` corre já com o tipo convertido.

In [ ]:
from typing import Literal

from pydantic import field_validator


class InscricaoCurso(BaseModel):
    email_contacto: str
    nivel: Literal["iniciante", "intermediario", "avancado"]

    @field_validator("email_contacto", mode="before")
    @classmethod
    def strip_email(cls, v: object) -> object:
        if isinstance(v, str):
            return v.strip().lower()
        return v


i = InscricaoCurso(email_contacto="  Ana.Exemplo@Escola.PT  ", nivel="iniciante")
print(i.email_contacto)

## 3. `model_validator` — regras entre campos

Útil quando uma restrição depende de **dois ou mais** atributos (ex.: desconto não pode exceder o preço base).

In [ ]:
from pydantic import model_validator


class OrcamentoTrilha(BaseModel):
    preco_base_eur: float = Field(ge=0)
    desconto_eur: float = Field(ge=0)

    @model_validator(mode="after")
    def desconto_coerente(self) -> "OrcamentoTrilha":
        if self.desconto_eur > self.preco_base_eur:
            raise ValueError("desconto_eur não pode ser superior a preco_base_eur")
        return self


o = OrcamentoTrilha(preco_base_eur=100.0, desconto_eur=25.0)
print("Total líquido:", o.preco_base_eur - o.desconto_eur)

try:
    OrcamentoTrilha(preco_base_eur=10.0, desconto_eur=50.0)
except ValidationError as e:
    print("Erro esperado:", e.errors()[0]["msg"])

## 4. Modelos aninhados e coleções

Compõe tipos complexos: uma **encomenda** contém várias **linhas**, cada uma com quantidade e preço unitário.

In [ ]:
class LinhaPedido(BaseModel):
    sku: str
    quantidade: int = Field(ge=1)
    preco_unitario_eur: float = Field(ge=0)

    @property
    def subtotal(self) -> float:
        return self.quantidade * self.preco_unitario_eur


class PedidoMaterial(BaseModel):
    referencia: str
    linhas: list[LinhaPedido]

    @property
    def total_eur(self) -> float:
        return sum(l.subtotal for l in self.linhas)


pedido = PedidoMaterial(
    referencia="PED-2026-0042",
    linhas=[
        LinhaPedido(sku="LIVRO-RAG", quantidade=2, preco_unitario_eur=29.9),
        LinhaPedido(sku="USB-C", quantidade=1, preco_unitario_eur=12.0),
    ],
)
print("Total:", round(pedido.total_eur, 2), "EUR")

## 5. Serialização: `model_dump`, `model_dump_json`, `model_validate_json`

- **`model_dump()`** — dicionário Python (pronto para `json.dumps` ou gravar em BD).
- **`model_dump_json()`** — string JSON.
- **`model_validate_json`** — reconstrói o modelo a partir de JSON (valida de novo).

In [ ]:
d = pedido.model_dump()
print("dict keys:", list(d.keys()))

js = pedido.model_dump_json(indent=2)
print(js[:200], "…")

copia = PedidoMaterial.model_validate_json(js)
assert copia.total_eur == pedido.total_eur

## 6. `ConfigDict` — comportamento do modelo

Exemplos comuns:

- `str_strip_whitespace=True` — remove espaços em strings ao validar.
- `extra="forbid"` — rejeita campos não declarados (útil contra payloads «a mais»).
- `frozen=True` — instância imutável (hashable se todos os campos forem hashable).

In [ ]:
from pydantic import ConfigDict


class TagMetadados(BaseModel):
    model_config = ConfigDict(extra="forbid", str_strip_whitespace=True)

    tema: str
    autor: str


t = TagMetadados(tema="  embeddings  ", autor="Equipa Curso")
print(t)

try:
    TagMetadados(tema="x", autor="y", campo_extra=1)
except ValidationError as e:
    print("extra forbid:", e.errors()[0]["type"])

## 7. JSON Schema (`model_json_schema`)

O schema descreve tipos e restrições num formato standard — ferramentas, OpenAPI e **structured output** de LLMs podem consumi-lo.

In [ ]:
schema = PedidoMaterial.model_json_schema()
print("title:", schema.get("title"))
print("properties:", list(schema.get("properties", {}).keys()))

## 8. Opcional — LangChain + Gemini com o **mesmo** modelo Pydantic

`ChatGoogleGenerativeAI.with_structured_output(MinhaClasse)` pede ao modelo uma resposta que **instancia** o schema. Útil para extrair campos de texto livre.

Se não tiveres API key, ignora a execução ou comenta a célula.

In [ ]:
from langchain_core.messages import HumanMessage
from langchain_google_genai import ChatGoogleGenerativeAI


class SumarioNoticia(BaseModel):
    """Schema para extracção simples — texto fictício."""

    titulo_curto: str = Field(description="Título com até 80 caracteres")
    entidades: list[str] = Field(description="Pessoas ou organizações mencionadas")
    sentimento: Literal["positivo", "neutro", "negativo"] = Field(description="Tom global")


texto_bruto = """
A Câmara Municipal anunciou hoje um programa de bolsas para formação em IA.
O presidente destacou o impacto positivo nas escolas da região.
"""

if not (os.environ.get("GOOGLE_API_KEY") or os.environ.get("GEMINI_API_KEY")):
    print("Sem GOOGLE_API_KEY — saltar invocação ao Gemini.")
else:
    modelo = (
        os.environ.get("GEMINI_MODEL_EX11")
        or os.environ.get("GEMINI_MODEL")
        or "gemini-2.0-flash"
    ).replace("models/", "")
    llm = ChatGoogleGenerativeAI(model=modelo, temperature=0.1)
    extrator = llm.with_structured_output(SumarioNoticia)
    out: SumarioNoticia = extrator.invoke(
        [HumanMessage(content="Extrai o schema a partir deste texto:\n" + texto_bruto)]
    )
    print(out.model_dump_json(indent=2, ensure_ascii=False))

## Referências

- [Documentação Pydantic v2](https://docs.pydantic.dev/latest/)
- [`docs/explicacao_teorica.md`](docs/explicacao_teorica.md) nesta pasta
- LangChain: *structured output* com modelos compatíveis